In [2]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:



try:
    # Load the actual data provided by user
    df = pd.read_csv('/content/drive/MyDrive/final_experiment_results.csv')

    # df = pd.read_csv('final_experiment_results.csv')
    df_frozen = pd.read_csv('/content/drive/MyDrive/frozen_eval_split.csv')


    audit_logs = []

    # 1. Check ID/Label alignment & Partitions
    # (Assuming both CSVs have the same sequential questions or a pubid to match,
    # we'll do a soft check on lengths and question overlap)
    if len(df) == 50 and len(df_frozen) == 50:
        audit_logs.append("[PASSED] Partition Length: Exact 50 records found in both evaluation and frozen splits.")
    else:
        audit_logs.append(f"[FAILED] Partition Length Mismatch: Eval({len(df)}) vs Frozen({len(df_frozen)}).")

    if all(df['question'] == df_frozen['question']):
        audit_logs.append("[PASSED] Alignment: All 50 queries align perfectly across methods.")
    else:
        audit_logs.append("[WARNING] Row sequence alignment mismatch. Attempting merge on 'question'.")

    # 2. Check for Missing values & Data Types
    missing_vals = df[['zs_plausibility', 'rag_plausibility', 'debate_plausibility']].isna().sum().sum()
    if missing_vals == 0:
        audit_logs.append("[PASSED] Missing Values: 0 NaN metrics detected.")
    else:
        audit_logs.append(f"[FAILED] Missing Values: {missing_vals} missing metric values detected.")

    # 3. Compute Metrics with Uncertainty (Std Dev)
    metrics = []
    for method, prefix in [('Zero-Shot Single LLM', 'zs'), ('Static Context RAG', 'rag'), ('2-Agent Debate', 'debate')]:
        scores = df[f'{prefix}_plausibility'].astype(float)
        tokens = df[f'{prefix}_input_tokens'].astype(float) + df[f'{prefix}_output_tokens'].astype(float)

        metrics.append({
            'Method': method,
            'Mean_Plausibility': scores.mean(),
            'Std_Plausibility': scores.std(),
            'Total_Tokens': tokens.sum(),
            'Mean_Tokens_Per_Query': tokens.mean(),
            'Plausibility_ROI': scores.mean() / (tokens.sum() / 1000)
        })

    metrics_df = pd.DataFrame(metrics)
    metrics_df.to_csv('evaluation_summary.csv', index=False)

    # 4. Statistical Testing (Wilcoxon signed-rank test for ordinal plausibility scores)
    # Check RAG vs Debate
    stat, p_value = stats.wilcoxon(df['rag_plausibility'], df['debate_plausibility'])
    stat_msg = f"Wilcoxon Signed-Rank Test (RAG vs. 2-Agent Debate): W={stat}, p-value={p_value:.4e}"
    if p_value < 0.05:
        stat_msg += " (Statistically Significant)"
    else:
        stat_msg += " (Not Significant)"

    # 5. Plotting (Dual Axis: Plausibility w/ Std Dev & Token Cost)
    plt.style.use('default')
    fig, ax1 = plt.subplots(figsize=(9, 6))

    sns.barplot(
        x='Method',
        y='Mean_Plausibility',
        data=metrics_df,
        color='steelblue',
        alpha=0.8,
        ax=ax1,
        capsize=0.1
    )

    # Manual error bars for standard deviation
    ax1.errorbar(
        x=np.arange(len(metrics_df)),
        y=metrics_df['Mean_Plausibility'],
        yerr=metrics_df['Std_Plausibility'],
        fmt='none',
        c='black',
        capsize=6,
        linewidth=2
    )

    ax1.set_ylabel('Mean Plausibility (1-10)', fontsize=12, color='black', fontweight='bold')
    ax1.set_ylim(0, 10.5)
    ax1.tick_params(axis='y', labelcolor='black')
    ax1.set_xlabel('')

    ax2 = ax1.twinx()
    sns.lineplot(
        x='Method',
        y='Total_Tokens',
        data=metrics_df,
        marker='o',
        markersize=10,
        linewidth=3,
        color='darkred',
        ax=ax2,
        sort=False
    )

    ax2.set_ylabel('Total Token Consumption', fontsize=12, color='darkred', fontweight='bold')
    ax2.tick_params(axis='y', labelcolor='darkred')

    # Set y-axis to log scale for tokens to show the magnitude difference clearly
    ax2.set_yscale('log')
    ax2.set_ylim(1000, 500000)

    plt.title('Performance and Cost Analysis:\nPlausibility vs. Token Consumption (Log Scale)', fontsize=14, pad=15)
    plt.grid(axis='y', linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.savefig('performance_comparison.png', dpi=300, bbox_inches='tight')
    plt.close()

    print("--- AUDIT LOGS ---")
    print("\n".join(audit_logs))
    print("\n--- STATISTICAL ANALYSIS ---")
    print(stat_msg)
    print("\n--- FINAL METRICS ---")
    print(metrics_df.to_string(index=False))

except Exception as e:
    print(f"Error occurred: {e}")

--- AUDIT LOGS ---
[PASSED] Partition Length: Exact 50 records found in both evaluation and frozen splits.
[PASSED] Alignment: All 50 queries align perfectly across methods.
[PASSED] Missing Values: 0 NaN metrics detected.

--- STATISTICAL ANALYSIS ---
Wilcoxon Signed-Rank Test (RAG vs. 2-Agent Debate): W=5.0, p-value=4.2268e-09 (Statistically Significant)

--- FINAL METRICS ---
              Method  Mean_Plausibility  Std_Plausibility  Total_Tokens  Mean_Tokens_Per_Query  Plausibility_ROI
Zero-Shot Single LLM               1.54          2.159081        2502.0                  50.04          0.615508
  Static Context RAG               2.14          2.962418        3320.0                  66.40          0.644578
      2-Agent Debate               8.58          2.821853      294079.0                5881.58          0.029176


In [6]:
# metrics_evaluation.py (Evaluation Code for AI Debugging Log)
import pandas as pd
import numpy as np
from scipy import stats


 # Load the actual data provided by user
df = pd.read_csv('/content/drive/MyDrive/final_experiment_results.csv')

    # df = pd.read_csv('final_experiment_results.csv')
df_frozen = pd.read_csv('/content/drive/MyDrive/frozen_eval_split.csv')

# 2. Strict Auditing
assert len(df) == 50 and len(df_frozen) == 50, "Partition mismatch"
assert all(df['question'] == df_frozen['question']), "Label alignment failed"
assert df[['zs_plausibility', 'rag_plausibility', 'debate_plausibility']].isna().sum().sum() == 0, "Missing metrics detected"

# 3. Compute Metrics
metrics = []
for method, prefix in [('Zero-Shot Single LLM', 'zs'), ('Static Context RAG', 'rag'), ('2-Agent Debate', 'debate')]:
    scores = df[f'{prefix}_plausibility'].astype(float)
    tokens = df[f'{prefix}_input_tokens'].astype(float) + df[f'{prefix}_output_tokens'].astype(float)

    metrics.append({
        'Method': method,
        'Mean_Plausibility': scores.mean(),
        'Std_Plausibility': scores.std(),
        'Total_Tokens': tokens.sum(),
        'Plausibility_ROI': scores.mean() / (tokens.sum() / 1000)
    })

metrics_df = pd.DataFrame(metrics)
metrics_df.to_csv('evaluation_summary.csv', index=False)

# 4. Statistical Testing
stat, p_value = stats.wilcoxon(df['rag_plausibility'], df['debate_plausibility'])
print(f"Wilcoxon p-value: {p_value:.4e}")

Wilcoxon p-value: 4.2268e-09
